In [73]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [74]:
expression_data = pd.read_csv('expression_data.csv')
gene_variances = expression_data.var(axis=1)

def knn_train_and_predict(num_genes: int):
    top_n_genes = gene_variances.sort_values(ascending=False).head(num_genes).index
    top_variable_expression = expression_data.loc[top_n_genes].T

    X = top_variable_expression
    y = pd.read_csv('../data/processed/group_annotation.csv', index_col=0)['Group']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=0, stratify=y
    )

    knn_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier())
    ])

    param_grid = {
        'knn__n_neighbors': range(1, 21, 2) # Test odd k values from 1 to 19
    }

    grid_search = GridSearchCV(
        estimator=knn_pipeline,
        param_grid=param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1 # Use all available cores
    )
    grid_search.fit(X_train, y_train)

    best_k = grid_search.best_params_['knn__n_neighbors']

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    test_accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return best_k, test_accuracy, report

### 1. Train KNN on Assignment 1 Groups

In [75]:
best_k, test_accuracy, report = knn_train_and_predict(5000)

print(f"Assignment 1 groups prediction accuracy with 5000 genes and best k={best_k}: {test_accuracy:.4f}")
print("\nClassification report:")
print(report)

Assignment 1 groups prediction accuracy with 5000 genes and best k=1: 0.9259

Classification report:
              precision    recall  f1-score   support

          T0       0.88      1.00      0.93        14
          T3       1.00      0.85      0.92        13

    accuracy                           0.93        27
   macro avg       0.94      0.92      0.93        27
weighted avg       0.94      0.93      0.93        27



### 2. Train KNN on Assignment 3 Affinity Propagation Clusters

In [76]:
# load affinity propagation clustering results from assignment 3
clusters = pd.read_csv('../results/cluster_results_ap.csv').set_index('Sample')
label_col = 'Cluster_AP'

X = top_variable_expression.copy()
common = X.index.intersection(clusters.index)
if len(common) == 0:
    raise ValueError("No sample IDs match between top_variable_expression.index and cluster_results_k3.csv Sample column.")
X = X.loc[common]
y = clusters.loc[common, label_col].astype(int)

# sanity check
print("Cluster counts:\n", y.value_counts())

# split
X_train_ap, X_test_ap, y_train_ap, y_test_ap = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(weights = 'distance'))
])

param_grid = {
    'knn__n_neighbors': range(1, 21, 2)
}

grid_search_ap = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_ap.fit(X_train_ap, y_train_ap)

best_model_ap = grid_search_ap.best_estimator_
y_pred_ap = best_model_ap.predict(X_test_ap)

print(f"\nBest k for affinity propagation clustering results classification: {grid_search_ap.best_params_['knn__n_neighbors']}")

Cluster counts:
 Cluster_AP
5    21
6    15
8    12
2    10
3     7
0     6
9     6
4     5
1     4
7     2
Name: count, dtype: int64


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(



Best k for affinity propagation clustering results classification: 5


In [77]:
test_accuracy_ap = accuracy_score(y_test_ap, y_pred_ap)
print(f"Assignment 3 clusters (affinity propagation) prediction accuracy: {test_accuracy_ap:.4f}")

print("\nK nearest neighbors classification report:")
print(classification_report(y_test_ap, y_pred_ap))
print("Confusion matrix:")
print(confusion_matrix(y_test_ap, y_pred_ap))
pd.DataFrame({'sample_id': X_test_ap.index, 'true_cluster': y_test_ap.values, 'pred_cluster_knn': y_pred_ap}).to_csv('../results/knn_multiclass_ap_predictions.csv', index=False)

Assignment 3 clusters (affinity propagation) prediction accuracy: 0.6111

K nearest neighbors classification report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       1.00      1.00      1.00         1
           2       0.00      0.00      0.00         2
           3       1.00      0.50      0.67         2
           4       1.00      1.00      1.00         1
           5       0.44      1.00      0.62         4
           6       1.00      0.33      0.50         3
           8       1.00      1.00      1.00         3
           9       0.00      0.00      0.00         1

    accuracy                           0.61        18
   macro avg       0.60      0.54      0.53        18
weighted avg       0.65      0.61      0.57        18

Confusion matrix:
[[0 0 0 0 0 1 0 0 0]
 [0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 0]
 [1 0 0 1 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 4 0 0 0]
 [0 0 0 0 0 2 1 0 0]
 [0 0 0 0 0 0

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

### 3. Train KNN on Different Numbers of Genes

In [78]:
for n in [10, 100, 1000, 10000]:
    best_k, test_accuracy, report = knn_train_and_predict(n)

    print(f"Assignment 1 groups prediction accuracy with {n} genes and best k={best_k}: {test_accuracy:.4f}")
    print("\nClassification report:")
    print(report)

Assignment 1 groups prediction accuracy with 10 genes and best k=11: 0.7778

Classification report:
              precision    recall  f1-score   support

          T0       0.79      0.79      0.79        14
          T3       0.77      0.77      0.77        13

    accuracy                           0.78        27
   macro avg       0.78      0.78      0.78        27
weighted avg       0.78      0.78      0.78        27

Assignment 1 groups prediction accuracy with 100 genes and best k=17: 0.8889

Classification report:
              precision    recall  f1-score   support

          T0       0.92      0.86      0.89        14
          T3       0.86      0.92      0.89        13

    accuracy                           0.89        27
   macro avg       0.89      0.89      0.89        27
weighted avg       0.89      0.89      0.89        27

Assignment 1 groups prediction accuracy with 1000 genes and best k=9: 0.7407

Classification report:
              precision    recall  f1-score 